In [ ]:
%pip install eyepop==3.12.0

In [ ]:
import getpass

EYEPOP_ACCOUNT_ID=input("Enter your Account UUID: ")
EYEPOP_API_KEY=getpass.getpass('Enter your API KEY: ')

In [ ]:
NAMESPACE_PREFIX="XXXXXXXXXXX" # Add your namespace-prefix here

### Define Your Ability

In [ ]:
from eyepop import EyePopSdk
from eyepop.data.data_types import InferRuntimeConfig, VlmAbilityGroupCreate, VlmAbilityCreate, TransformInto
from eyepop.worker.worker_types import CropForward, ForwardComponent, FullForward, InferenceComponent, Pop
import json


ability_prototypes = [
    VlmAbilityCreate(
        name=f"{NAMESPACE_PREFIX}.describe.read-dial",
        description="Read a dial/meter",
        worker_release="qwen3-instruct",
        text_prompt="""
          Analyze the provided image of a meter or dial and extract the current numerical reading.

          INSTRUCTIONS:
          1. LOCATE THE INDICATOR TIP: Find the long, thin, pointed end of the indicator needle.
          2. LOCATE THE OUTER SCALE: Look ONLY at the outermost ring of measurement numbers and tick marks. If there are multiple scales (e.g., black numbers on the outside, red numbers on the inside), strictly use the outermost scale.
          3. IGNORE CENTER TEXT: Do not read any text, weight capacities, or secondary numbers printed in the center of the dial face (e.g., ignore text like "10g" or "20kg").
          4. DETERMINE INTERVALS: Identify the printed whole numbers immediately before and after the needle tip. Count the minor tick marks between them to figure out the mathematical value of each space (e.g., if there are 4 spaces between 1 and 2, each tick mark represents 0.25).

          CRITICAL FORMATTING:
          Output ONLY the final calculated numerical value for the outermost scale. Do not include units of measurement, symbols, markdown formatting, or any conversational explanation.
        """,
        transform_into=TransformInto(),
        config=InferRuntimeConfig(
            max_new_tokens=150,
            image_size=1000
        ),
        is_public=False
    )
]

### Create Your Ability

In [ ]:
with EyePopSdk.dataEndpoint(api_key=EYEPOP_API_KEY, account_id=EYEPOP_ACCOUNT_ID) as endpoint:
    for ability_prototype in ability_prototypes:
        ability_group = endpoint.create_vlm_ability_group(VlmAbilityGroupCreate(
            name=ability_prototype.name,
            description=ability_prototype.description,
            default_alias_name=ability_prototype.name,
        ))
        ability = endpoint.create_vlm_ability(
            create=ability_prototype,
            vlm_ability_group_uuid=ability_group.uuid,
        )
        ability = endpoint.publish_vlm_ability(
            vlm_ability_uuid=ability.uuid,
            alias_name=ability_prototype.name,
        )
        ability = endpoint.add_vlm_ability_alias(
            vlm_ability_uuid=ability.uuid,
            alias_name=ability_prototype.name,
            tag_name="latest"
        )
        print(f"created ability {ability.uuid} with alias entries {ability.alias_entries}")

### Evalulate on a Single Image

In [ ]:
from pathlib import Path

pop = Pop(components=[
   InferenceComponent(
       ability=f"{NAMESPACE_PREFIX}.describe.read-dial:latest"
   )
])

with EyePopSdk.workerEndpoint(api_key=EYEPOP_API_KEY) as endpoint:
   endpoint.set_pop(pop)
   sample_img_path = Path("/content/sample_img.png")
   job = endpoint.upload(sample_img_path)
   while result := job.predict():
      print(json.dumps(result, indent=2))

print("Done")